In [4]:
import streamlit as st
import pandas as pd
import numpy as np
import re
import json
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import networkx as nx

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append(os.path.abspath(os.path.join('..')))
from src import plots
from src import ml_processing

c:\Users\CABOULOU\OneDrive - Volvo Cars\Bureau\trustpilot v2\sentiment-analysis-reviews\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CABOULOU\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\CABOULOU\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Load data

In [5]:
def loadData(uploaded_file):
    if uploaded_file is not None:
        return pd.read_csv(uploaded_file)
    return None

def extractPrefix(file_name):
    # Split the filename and extract the part before "_ml"
    return file_name.split('_ml')[0]

def loadJson(file_path):
    with open(file_path, 'r') as file:
        return json.load(file)

def reFormatEmbeddings(embedding_str):
    cleaned_str = re.sub(r'[\[\]\n]', '', embedding_str)
    embedding_list = list(map(float, cleaned_str.split()))
    return np.array(embedding_list, dtype=np.float32)
    return embedding_str

processed_path = '../data/processed/'
raw_path = '../data/raw/'

In [6]:
uploaded_file = 'hd_ml_processed_reviews.csv'

## Load all necessary data
# Load reviews data and extract place from the file name
reviews = loadData(processed_path + uploaded_file)
if 'embedding' in reviews.columns:
    # Convert embeddings from string to list of floats
    reviews['embedding'] = reviews['embedding'].apply(reFormatEmbeddings)

file_name = uploaded_file
place = extractPrefix(file_name)

# Paths for the JSON and additional CSV files
general_insights_file = os.path.join(processed_path, f"{place}_general_insights.json")
worst_periods_file = os.path.join(processed_path, f"{place}_worst_periods_insights.json")
sample_reviews_file = os.path.join(processed_path, f"{place}_sample_selected_reviews.csv")
resume_file = os.path.join(raw_path, f"resumme_{place}.csv")

# Load "place"_general_insights.json into a dictionary
if os.path.exists(general_insights_file):
    general_insights = loadJson(general_insights_file)

# Load "place"_worst_periods_insights.json into a dictionary
if os.path.exists(worst_periods_file):
    worst_periods_insights = loadJson(worst_periods_file)

# Load "place"_sample_selected_reviews.csv into a DataFrame
if os.path.exists(sample_reviews_file):
    sample_reviews = pd.read_csv(sample_reviews_file)

# Load resumme_"place".csv from ./data/raw into a DataFrame
if os.path.exists(resume_file):
    resume = pd.read_csv(resume_file)

### Dev

In [7]:
display(sample_reviews.sample(3))

# best_reviews
best_reviews = sample_reviews[sample_reviews['sample_type'] == 'best_reviews_sample'][['date', 'rating_score','review', 'food_score', 'service_score', 'atmosphere_score', 'meal_type']]
best_reviews.rename(columns = {'review':'Review', 'rating_score':'Rating', 'meal_type':'Meal','food_score':'Food', 'service_score':'Service', 'atmosphere_score':'Ambient', 'date':'Date'}, inplace = True)

# worst_reviews
worst_reviews = sample_reviews[sample_reviews['sample_type'] == 'worst_reviews_sample'][['date', 'rating_score','review', 'food_score', 'service_score', 'atmosphere_score', 'meal_type']]
worst_reviews.rename(columns = {'review':'Review', 'rating_score':'Rating', 'meal_type':'Meal','food_score':'Food', 'service_score':'Service', 'atmosphere_score':'Ambient', 'date':'Date'}, inplace = True)

,review_id,review,local_guide_reviews,rating_score,service,meal_type,price_per_person_category,food_score,service_score,atmosphere_score,...,cleaned_review,vader_sentiment,sentiment_label,embedding,pca_cluster,umap_cluster,month,year,total_score,sample_type
30,8.0,"Viví en este barrio en los 80 , he vuelto a es...",4.0,5.0,Comí allí,Cena,30-40 €,4.000,5.000,5.000,...,viví barrio 80 volver emblemático café quedar ...,0.0,positive,[-1.07015148e-02 -6.65618122e-01 9.12529603e-...,0.0,0.0,2024-05,2024.0,5.93,recent_best_reviews
277,245.0,Súper hamburguesas y una gran selección de beb...,85.0,5.0,NaN,NaN,NaN,0.874,0.902,0.894,...,súper hamburguesa gran selección bebida,0.0,positive,[-6.03224337e-01 -5.53049982e-01 7.01106712e-...,-1.0,0.0,2018-01,2018.0,5.18,best_reviews_sample
170,15.0,La comida me resultó bastante mala. Los canelo...,102.0,1.0,Comí allí,Comida,10-20 €,1.000,4.000,1.000,...,comida resultar bastante malo canelón parecer ...,0.0,negative,[-1.88059866e-01 -2.77377814e-01 2.07078621e-...,0.0,0.0,2024-08,2024.0,1.40,recent_worst_reviews


In [8]:
general_insights

{'best': ['The quality of the hamburgers is consistently praised by customers.',
  'The overall dining experience is enhanced by a pleasant atmosphere and good service.',
  'Many customers appreciate the reasonable pricing and value for money.'],
 'worst': ['Some customers express disappointment with the service, noting it can be slow or subpar.',
  'Issues regarding the quality of food items, such as cold dishes or poor presentation, have been reported.',
  'The menu options available for brunch have been criticized as lacking in variety.'],
 'improve': ['Enhance service efficiency to ensure quicker response times and improve overall customer satisfaction.',
  'Reassess the quality control measures in the kitchen to guarantee that all dishes meet high standards before serving.',
  'Expand the brunch menu to include a wider variety of options to cater to different customer preferences.']}

In [9]:
worst_periods_insights

{'2023-11': {'problems': ['The atmosphere was often perceived as uncomfortable during the evening.',
   'Customers felt dissatisfaction with the service provided at the cafeteria.'],
  'improve': ['Enhance the overall comfort of the seating arrangements for evening customers.',
   'Improve service efficiency in the cafeteria to address customer concerns.']},
 '2024-02': {'problems': ['There were complaints regarding the overall quality of service.',
   'Customers felt let down by their experiences when placing orders.'],
  'improve': ['Invest in training staff to improve service interactions with customers.',
   'Review the order processing system to ensure timely and accurate service.']},
 '2024-03': {'problems': ['The noise level in the venue was frequently mentioned as disruptive.',
   'Customers expressed frustration with certain aspects of the drink menu.'],
  'improve': ['Implement measures to reduce noise levels during peak hours.',
   'Consider expanding the drink menu to inclu

In [10]:
display(sample_reviews.sample(3))

,review_id,review,local_guide_reviews,rating_score,service,meal_type,price_per_person_category,food_score,service_score,atmosphere_score,...,cleaned_review,vader_sentiment,sentiment_label,embedding,pca_cluster,umap_cluster,month,year,total_score,sample_type
280,237.0,jugando es el mejor,21.0,5.0,NaN,NaN,NaN,0.874,0.902,0.894,...,jugar mejor,0.0,positive,[-1.52121827e-01 -6.07512116e-01 -1.14238769e-...,0.0,0.0,2024-05,2024.0,5.18,best_reviews_sample
108,90.0,Excelente decoración conservando el estilo tar...,148.0,5.0,NaN,NaN,NaN,0.874,0.902,0.894,...,excelente decoración conservar estilo tardofra...,0.0,positive,[-5.35155796e-02 -4.56613123e-01 5.00874281e-...,0.0,0.0,2021-01,2021.0,5.18,recent_best_reviews
298,273.0,¡Excelente!,107.0,5.0,NaN,NaN,NaN,0.874,0.902,0.894,...,excelente,0.0,positive,[ 0.21421671 -0.31342492 0.8391285 0.137127...,0.0,0.0,2022-01,2022.0,5.18,best_reviews_sample


In [11]:
sample_reviews.groupby('sample_type').count()

,review_id,review,local_guide_reviews,rating_score,service,meal_type,price_per_person_category,food_score,service_score,atmosphere_score,...,avg_price_per_person,cleaned_review,vader_sentiment,sentiment_label,embedding,pca_cluster,umap_cluster,month,year,total_score
sample_type,,,,,,,,,,,,,,,,,,,,,
best_reviews_sample,168,168,168,168,42,46,43,168,168,168,...,43,167,168,168,168,168,168,167,167,168
low_score_reviews,0,4,0,4,0,0,0,0,0,0,...,0,0,0,0,0,0,0,4,0,0
recent_best_reviews,168,168,168,168,42,46,43,168,168,168,...,43,167,168,168,168,168,168,167,167,168
recent_worst_reviews,18,18,18,18,3,3,2,18,18,18,...,2,18,18,18,18,18,18,17,17,18
worst_reviews_sample,18,18,18,18,3,3,2,18,18,18,...,2,18,18,18,18,18,18,17,17,18


In [12]:
period_reviews = sample_reviews[(sample_reviews['month'] == '2024-08') & (sample_reviews['sample_type'] == 'low_score_reviews')][['date', 'rating_score', 'review', 'food_score', 'service_score', 'atmosphere_score', 'meal_type']]
period_reviews

,date,rating_score,review,food_score,service_score,atmosphere_score,meal_type
375,NaN,1.0,La comida me resultó bastante mala. Los canelo...,NaN,NaN,NaN,NaN


### Plots

In [13]:
def plotTrend(reviews, label_mapping, app=False, filter_min=None, filter_max=None):
    # Convert date column to datetime format and create additional time columns
    reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')
    reviews['month'] = reviews['date'].dt.to_period('M')

    # Filter data for the last periods based on filter_min and filter_max
    limit_date = reviews['date'].max()
    if filter_min is None and filter_max is None:
        # If both filters are None, select data from the last year
        start_date = limit_date - pd.DateOffset(years=1)
        selected_reviews = reviews[(reviews['date'] >= start_date) & (reviews['date'] <= limit_date)]
    else:
        # Apply the filters if provided
        selected_reviews = reviews
        if filter_min is not None:
            selected_reviews = selected_reviews[selected_reviews['date'] >= filter_min]
        if filter_max is not None:
            selected_reviews = selected_reviews[selected_reviews['date'] <= filter_max]

    # Compute averages for the required periods using label_mapping keys
    columns_to_average = list(label_mapping.keys())
    monthly_avg_scores = selected_reviews.groupby('month')[columns_to_average].mean()
    
    # Create a figure to plot the trends
    fig = make_subplots(rows=1, cols=1)
    
    # Update the axis labels for each score to be more readable
    colors = ['#32CD32', 'rgba(31, 119, 180, 0.8)', 'rgba(107, 174, 214, 0.8)', 'rgba(158, 202, 225, 0.8)'] 
    for i, column in enumerate(monthly_avg_scores.columns):
        label = label_mapping[column]
        fig.add_trace(
            go.Scatter(x=monthly_avg_scores.index.astype(str), y=monthly_avg_scores[column],
                       mode='lines+markers', name=label, 
                       text=[f"{label} - {val:.2f}" for val in monthly_avg_scores[column]], 
                       hoverinfo="text", line=dict(color=colors[i], width=3 if i == 0 else 2)),
            row=1, col=1)


    # Analyze low scores and find high score
    _, low_score_periods = ml_processing.analyzeLowScores(reviews, 'rating_score', num_periods=3)
    high_score_period = monthly_avg_scores['rating_score'].idxmax()
    high_score_value = monthly_avg_scores['rating_score'].max()
    
    # Add annotations for low scores
    for i in range(len(low_score_periods)):
        if i > 0 and low_score_periods[i] - low_score_periods[i - 1] == 1:
            # If two periods are contiguous, combine them in one annotation
            fig.add_annotation(x=str(low_score_periods[i]), y=monthly_avg_scores.loc[low_score_periods[i], 'rating_score'] + 0.5,
                               text=f"Drop in {low_score_periods[i - 1].strftime('%B')} & {low_score_periods[i].strftime('%B')}",
                               showarrow=True, arrowhead=2, ax=0, ay=-40, row=1, col=1)
        elif i == 0 or low_score_periods[i] - low_score_periods[i - 1] != 1:
            fig.add_annotation(x=str(low_score_periods[i]), y=monthly_avg_scores.loc[low_score_periods[i], 'rating_score'] + 0.5,
                               text=f"Drop in {low_score_periods[i].strftime('%B')}",
                               showarrow=True, arrowhead=2, ax=0, ay=-40, row=1, col=1)
    
    # Add annotation for high score
    fig.add_annotation(x=str(high_score_period), y=high_score_value - 0.3,
                       text=f"High in {high_score_period.strftime('%B')}",
                       showarrow=True, arrowhead=2, ax=0, ay=40, row=1, col=1)

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False, title_text='Average Score')
    fig.update_layout(showlegend=False, 
                    #title="Rating Trends",
                    #title_font=dict(size=28),
                    margin=dict(l=50, r=50, t=100, b=50),
                    paper_bgcolor="white",
                    height=400, width=1200)
    
    # Show or return the figure depending on the context
    if app:
        return fig
    else:
        fig.show()

In [14]:
!pip install nbformat



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
label_mapping = {
    'rating_score': 'Rating',
    'food_score': 'Food',
    'service_score': 'Service',
    'atmosphere_score': 'Ambient'
}

plotTrend(reviews, label_mapping, app=False)
#plotTrend(reviews, label_mapping, app=False, filter_min="2024-01-01", filter_max="2024-05-01")

In [16]:
_, low_score_periods = ml_processing.analyzeLowScores(reviews, 'rating_score', num_periods=3)

In [17]:
worst_periods_insights

{'2023-11': {'problems': ['The atmosphere was often perceived as uncomfortable during the evening.',
   'Customers felt dissatisfaction with the service provided at the cafeteria.'],
  'improve': ['Enhance the overall comfort of the seating arrangements for evening customers.',
   'Improve service efficiency in the cafeteria to address customer concerns.']},
 '2024-02': {'problems': ['There were complaints regarding the overall quality of service.',
   'Customers felt let down by their experiences when placing orders.'],
  'improve': ['Invest in training staff to improve service interactions with customers.',
   'Review the order processing system to ensure timely and accurate service.']},
 '2024-03': {'problems': ['The noise level in the venue was frequently mentioned as disruptive.',
   'Customers expressed frustration with certain aspects of the drink menu.'],
  'improve': ['Implement measures to reduce noise levels during peak hours.',
   'Consider expanding the drink menu to inclu

In [18]:
filter_min = pd.to_datetime('2024/01/29')
filter_max = pd.to_datetime('2024/09/29')
worst_periods_insights_filtered = {k: v for k, v in worst_periods_insights.items() if filter_min <= pd.to_datetime(k) <= filter_max} if filter_min is not None and filter_max is not None else worst_periods_insights
worst_periods_insights_filtered

{'2024-02': {'problems': ['There were complaints regarding the overall quality of service.',
   'Customers felt let down by their experiences when placing orders.'],
  'improve': ['Invest in training staff to improve service interactions with customers.',
   'Review the order processing system to ensure timely and accurate service.']},
 '2024-03': {'problems': ['The noise level in the venue was frequently mentioned as disruptive.',
   'Customers expressed frustration with certain aspects of the drink menu.'],
  'improve': ['Implement measures to reduce noise levels during peak hours.',
   'Consider expanding the drink menu to include quieter options.']},
 '2024-08': {'problems': ['Some customers found the seating arrangements inadequate.',
   'Concerns were raised regarding the cleanliness and presentation of tables.'],
  'improve': ['Re-evaluate the layout of tables to enhance comfort and accessibility.',
   'Ensure more consistent cleaning and presentation of dining areas.']}}

In [19]:
from datetime import datetime
dates = list(worst_periods_insights.keys())
dates = [datetime.strptime(date, '%Y-%m') for date in dates]
limit_date = max(dates)

start_date = filter_min if filter_min is not None else (limit_date - pd.DateOffset(years=1))
end_date = filter_max if filter_max is not None else limit_date

# Asegurarse de que start_date y end_date sean objetos datetime
start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)

# Filtrar el diccionario usando los límites de fecha
filtered_insights = {date: data for date, data in worst_periods_insights.items()
                     if start_date <= datetime.strptime(date, '%Y-%m') <= end_date}

print(start_date.strftime('%Y-%m'))
print(end_date.strftime('%Y-%m'))
print(filtered_insights)



2024-01
2024-09
{'2024-02': {'problems': ['There were complaints regarding the overall quality of service.', 'Customers felt let down by their experiences when placing orders.'], 'improve': ['Invest in training staff to improve service interactions with customers.', 'Review the order processing system to ensure timely and accurate service.']}, '2024-03': {'problems': ['The noise level in the venue was frequently mentioned as disruptive.', 'Customers expressed frustration with certain aspects of the drink menu.'], 'improve': ['Implement measures to reduce noise levels during peak hours.', 'Consider expanding the drink menu to include quieter options.']}, '2024-08': {'problems': ['Some customers found the seating arrangements inadequate.', 'Concerns were raised regarding the cleanliness and presentation of tables.'], 'improve': ['Re-evaluate the layout of tables to enhance comfort and accessibility.', 'Ensure more consistent cleaning and presentation of dining areas.']}}


In [20]:
resume

,stars,reviews
0,5,2290
1,4,1308
2,3,396
3,2,132
4,1,128
